In [ ]:
import pandas as pd
import re
from bs4 import BeautifulSoup

liste_soft_skills = [
    "communication", "communiquer", "esprit d'équipe", "travail en équipe", "travailler en équipe",
    "autonomie","autonome", "rigueur", "organisation", "organisé", "créativité", "créatif",
    "leadership", "gestion de projet","innovation", "résolution de problèmes", "adaptabilité",
    "force de proposition", "proactivité", "relationnel", "initiative", "polyvalence", "flexibilité", 
    "sens de l'organisation", "réactivité", "esprit d'analyse"
]
liste_secteurs = [
    "automobile", "aéronautique", "métallurgie", "btp", "construction",
    "énergie", "agroalimentaire", "ferroviaire", "naval", "chimie",
    "plasturgie", "minier", "logistique", "bureau d'études",
    "pharmaceutique", "procédés", "recherche scientifique"
]

with open("outerhtml.txt", "r", encoding="utf-8") as f:                    # Lecture et analyse du fichier HTML
    html = f.read()
soup = BeautifulSoup(html, "html.parser")

liste_offres = []
mots_cles = ["technicien", "ingénieur", "maintenance", "mécanicien", "mecanique", "production"]       # Mots-clés utilisés pour filtrer les offres pertinentes

def nettoyer_texte(texte):
    return re.sub(r"[;\n\r]+", " ", texte).strip()                                       
# Fonctions d'analyse du texte
def detecter_langues(texte):
    texte = texte.lower()
    langues = []
    if "français" in texte or "francais" in texte:
        langues.append("Français")
    if "anglais" in texte or "english" in texte:
        langues.append("Anglais")
    if "arabe" in texte:
        langues.append("Arabe")
    return ", ".join(langues) if langues else "Non spécifié"

def detecter_etude(texte):
    texte = texte.lower()
    diplome = ["master", "bac+5", "bac+3", "bac+2", "licence", "deug", "dut", "bts"]
    for x in diplome:
        if x in texte:
            return x
    else:
        return "Non spécifié" 

def detecter_secteurs(texte):
    texte = texte.lower()
    secteurs_trouves = []
    for secteur in liste_secteurs:
        if secteur.lower() in texte:
            if secteur.capitalize() not in secteurs_trouves:                    
                secteurs_trouves.append(secteur.capitalize())
    if secteurs_trouves:
        return ", ".join(secteurs_trouves)
    else:
        return "Non spécifié"
    
def detecter_soft_skills(texte):
    texte = texte.lower()
    soft_skills_trouvees = []
    for skill in liste_soft_skills:
        if skill in texte:
            if skill.capitalize() not in soft_skills_trouvees:                     # Evite les doublons
                soft_skills_trouvees.append(skill.capitalize())
    if soft_skills_trouvees :
        return ", ".join(soft_skills_trouvees)
    else:
        return "Non spécifié"
# Fonctions d'extraction HTML
def detecter_contrat(bloc):
    for li in bloc.find_all("li"):
        texte_li = li.get_text(strip=True)
        if "Contrat proposé" in texte_li:
            balise_strong = li.find("strong")
            if balise_strong is not None:
                return balise_strong.get_text(strip=True)
    return "Non spécifié"

def detecter_experience(bloc):
    for li in bloc.find_all("li"):
        texte_li = li.get_text(strip=True)
        if "Niveau d'expérience" in texte_li:
            balise_strong = li.find("strong")
            if balise_strong is not None:
                return balise_strong.get_text(strip=True)
    return "Non spécifié"

def detecter_competences(bloc):
    for li in bloc.find_all("li"):
        texte_li = li.get_text(strip=True)
        if "Compétences clés" in texte_li:
            balise_strong = li.find("strong")
            if balise_strong is not None:
                return balise_strong.get_text(strip=True)
    return "Non spécifié"
#Traitement du titre/ville
def separer_titre_ville(titre):
    if " - " in titre:
        poste, ville = titre.rsplit(" - ", 1)                
        return poste.strip(), ville.strip()
    else:
        return titre.strip(), "Non spécifié"


for tag in soup.find_all("a"):                                      # Parcours des liens des offres d'emploi
    titre = tag.text.strip()
    if not (5 <= len(titre) <= 100):
        continue
    titre_lower = titre.lower()
    if not any(m in titre_lower for m in mots_cles):
        continue
    
    bloc = tag.find_parent("div")                                            # Retrouve le bloc contenant les offres
    if bloc is None:
        continue

    titre_poste, ville_titre = separer_titre_ville(titre) 

    texte_bloc = bloc.text.lower()
    entreprise_tag = bloc.find("a", class_="company-name") or bloc.find("div", class_="company")
    entreprise = entreprise_tag.text.strip() if entreprise_tag else "Non spécifié"

    langue = "Non spécifié"
    experience = "Non spécifié"
    for elem in bloc.find_all(["td", "li", "div", "span"]):            # Recherche des informations dans les différentes balises du bloc
        texte_elem = elem.text.strip().lower()
        if "langue" in texte_elem:
            langue = detecter_langues(elem.text)
       

    offre = {"Titre du poste": nettoyer_texte(titre_poste),
             "Entreprise": nettoyer_texte(entreprise),
             "Ville": ville_titre,
             "Secteur d'activité": detecter_secteurs(texte_bloc),
             "Contrat proposé": detecter_contrat(bloc),
             "Niveau d'étude": detecter_etude(texte_bloc).capitalize(),
             "Expérience": detecter_experience(bloc).strip("Expérience"),
             "Compétences techniques": detecter_competences(bloc).replace(" - ", ", "),
             "Langues": langue,
             "Soft skills": detecter_soft_skills(texte_bloc),
             "Lien de l'offre": tag.get("href", "") if tag.get("href", "").startswith("http") else "https://www.emploi.ma" + tag.get("href", ""),}
    liste_offres.append(offre)

# Exportation des offres vers un fichier CSV
if liste_offres:
    df = pd.DataFrame(liste_offres)
    df.to_csv("offres_emploima.csv", index=False, sep=",", encoding="utf-8-sig")
    print(f"Extraction: {len(df)} offres")

Extraction: 9 offres


In [3]:
import pandas as pd
df = pd.read_csv("offres_emploima.csv", sep=",")
df.head()

,Titre du poste,Entreprise,Ville,Secteur d'activité,Contrat proposé,Niveau d'étude,Expérience,Compétences techniques,Langues,Soft skills,Lien de l'offre
0,"Production, maintenance, qualité (6) Apply Pro...",Tectra Casa Tertiaire – Recrutement,Non spécifié,"Automobile, Aéronautique, Métallurgie, Btp, Co...",CDI,Bac+5,Débutant < 2 ans et plus,"Communication, Électromécanique, Génie Électri...","Français, Anglais, Arabe","Communication, Communiquer, Organisation, Gest...",https://www.emploi.ma/recherche-jobs-maroc/g%C...
1,Chargé de Maintenance et Entretien (mécanique),Tectra Casa Tertiaire – Recrutement,Casablanca,Btp,CDI,Bac+5,Débutant < 2 ans et plus,"Communication, Électromécanique, Génie Électri...",Non spécifié,Communication,https://www.emploi.ma/offre-emploi-maroc/charg...
2,Technicien de Maintenance Industrielle H/F,Crit Tanger,Kénitra,Non spécifié,Intérim,Bac+2,entre 2 ans et 5 ans,"Automatisme, Câblage, Électricité Industrielle...",Non spécifié,Non spécifié,https://www.emploi.ma/offre-emploi-maroc/techn...
3,Responsable Maintenance,Non spécifié,Mohammedia,Construction,CDI,Bac+5,entre 2 ans et 5 ans,"Électromécanique, Électrotechnique, Génie Élec...",Non spécifié,Non spécifié,https://www.emploi.ma/offre-emploi-maroc/respo...
4,Ingénieur Spécialisé en Instrumentation et Sys...,Expert Eye Engineering,Casablanca,Btp,CDI,Bac+5,entre 5 ans et 10 ans & Expérience > 10 ans,"API, BTP, Génie Électrique, Instrumentation",Non spécifié,Non spécifié,https://www.emploi.ma/offre-emploi-maroc/ingen...
